# Swedish electricity: cleaning prices, production and temperature

This notebook turns sixteen raw files, from five sources, into five tidy CSV files. Nothing is merged
here. Each source is cleaned on its own and written to its own file, so that any
later analysis can join them on whatever keys it needs.

## The data

| # | What it is | Where it came from | Shape of the raw file |
|---|---|---|---|
| 1 | Day-ahead electricity spot prices for the four Swedish bidding zones SE1 to SE4, in EUR/MWh | Nord Pool, via an API export | long: one row per zone and time step |
| 2 | Electricity production and usage in GWh, by category and bidding zone | SCB, Statistics Sweden | wide: one column per month |
| 3 | Hourly air temperature in degrees Celsius, one file per zone reference station | SMHI open data | semicolon separated, station metadata block on top |
| 4 | Cross-border physical flows between Sweden and six neighbours, in GW | Energy-Charts, Fraunhofer ISE (CC BY 4.0) | five JSON files of parallel arrays, one per year |
| 5 | National hourly generation by source, and load, in MW | Energy-Charts, Fraunhofer ISE (CC BY 4.0) | five JSON files of parallel arrays, one per year |

The SMHI stations stand in for their zones: Lulea-Kallax for SE1, Ostersund-Froson
for SE2, Stockholm-Arlanda for SE3 and Helsingborg A for SE4.

File paths below are relative to the repository root. Run from the project root or `notebooks/`.

## Reference period

Everything is checked against **2021-01-01 to 2025-12-31**. Timestamps are kept in
**UTC** throughout, which matters because the raw price file also carries Swedish
local time, and local time is not a unique key twice a year (see section 1).

## Outputs

| File | Columns | Grain |
|---|---|---|
| `data/clean/clean_prices_hourly.csv` | `ts_utc, zone, price_eur_mwh` | one row per zone and UTC hour |
| `data/clean/clean_scb_monthly.csv` | `category, zone, month, value_gwh` | one row per category, zone and month |
| `data/clean/clean_temperature_hourly.csv` | `ts_utc, zone, temp_c, quality` | one row per station and UTC hour |
| `data/clean/clean_flows_hourly.csv` | `ts_utc, denmark, finland, germany, lithuania, norway, poland, sum_gw` | one row per UTC hour, national |
| `data/clean/clean_generation_hourly.csv` | `ts_utc` plus eleven generation, load and renewable-share series | one row per UTC hour, national |

Every step that removes rows prints how many went and why. Run the cells top to
bottom; the notebook only reads the raw files and writes the five outputs above.

## Setup

In [1]:
from pathlib import Path

# Resolve paths from the repository root when launched here or from notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'data' / 'raw').is_dir() and (path / 'notebooks').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the project root or notebooks/ folder.')

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

CLEAN_DIR = PROJECT_ROOT / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

import os

import pandas as pd

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)
print('pandas', pd.__version__)
print('working directory:', os.getcwd())

pandas 3.0.3
working directory: C:\Users\ewuzi\Downloads\Project\notebooks


In [2]:
# The source CSV files in data/raw/. Everything below finds its input by
# pattern rather than by a fixed name, because these files have been renamed once
# already.
raw_files = sorted(RAW_DIR.glob('*.csv'))
for f in raw_files:
    print(f'{os.path.getsize(f) / 1e6:7.2f} MB   {f}')

   0.01 MB   C:\Users\ewuzi\Downloads\Project\data\raw\scb_production_consumption_monthly_2021_2026.csv.csv
   1.17 MB   C:\Users\ewuzi\Downloads\Project\data\raw\SE1 Luleå-Kallax.csv
   1.16 MB   C:\Users\ewuzi\Downloads\Project\data\raw\SE2 Östersund-Frösön.csv
   1.16 MB   C:\Users\ewuzi\Downloads\Project\data\raw\SE3 Stockholm-Arlanda.csv
   1.16 MB   C:\Users\ewuzi\Downloads\Project\data\raw\SE4 Helsingborg A.csv
  12.65 MB   C:\Users\ewuzi\Downloads\Project\data\raw\spot_prices_se1_se4_2021_2025.csv.csv


In [3]:
# One reference grid, used by every coverage check in the notebook.
GRID_START = '2021-01-01 00:00'
GRID_END = '2025-12-31 23:00'
HOURLY_GRID = pd.date_range(GRID_START, GRID_END, freq='h')
MONTHLY_GRID = pd.date_range('2021-01-01', '2025-12-01', freq='MS')
print(f'hourly grid:  {len(HOURLY_GRID):,} hours, {HOURLY_GRID[0]} to {HOURLY_GRID[-1]}')
print(f'monthly grid: {len(MONTHLY_GRID)} months, {MONTHLY_GRID[0].date()} to {MONTHLY_GRID[-1].date()}')

hourly grid:  43,824 hours, 2021-01-01 00:00:00 to 2025-12-31 23:00:00
monthly grid: 60 months, 2021-01-01 to 2025-12-01


In [4]:
def missing_hours(frame, ts_col='ts_utc', zone_col='zone'):
    """Hours of the reference grid with no observation, counted per zone."""
    out = {}
    for zone, part in frame.groupby(zone_col):
        present = pd.DatetimeIndex(part[ts_col].unique())
        out[zone] = int(len(HOURLY_GRID.difference(present)))
    return out

---
# 1. Nord Pool spot prices

Day-ahead prices for SE1 to SE4. Each row is one price for one zone at one time
step. The file carries both Swedish local time and UTC, an `_id` from the source
system, and a `price_unit` column.

Three things need attention:

1. **Two id-like columns carry nothing.** `_id` is a row id from the export and
   `price_unit` is the same string on every row.
2. **Duplicates depend on which timestamp you key on.** On Swedish local time there
   look to be many more duplicates than there really are.
3. **The resolution changes partway through.** The feed switches from hourly to
   15 minute values in October 2025, so the tail of the file is four times as dense
   as the head.

In [5]:
price_file = next(f for f in raw_files if 'spot' in f.name.lower())
print(price_file)

C:\Users\ewuzi\Downloads\Project\data\raw\spot_prices_se1_se4_2021_2025.csv.csv


In [6]:
prices_raw = pd.read_csv(price_file, encoding='utf-8-sig')
print(prices_raw.shape)
prices_raw.head()

(194804, 6)


,_id,start_time_sweden,start_time_utc,bidding_zone,price,price_unit
0,78204,2025-12-31T23:45:00,2025-12-31T22:45:00,SE1,15.26,EUR-MWh
1,190762,2025-12-31T23:45:00,2025-12-31T22:45:00,SE3,30.04,EUR-MWh
2,243514,2025-12-31T23:45:00,2025-12-31T22:45:00,SE2,20.22,EUR-MWh
3,767620,2025-12-31T23:45:00,2025-12-31T22:45:00,SE4,33.70,EUR-MWh
4,166666,2025-12-31T23:30:00,2025-12-31T22:30:00,SE3,33.00,EUR-MWh


In [7]:
prices_raw.dtypes

_id                    int64
start_time_sweden        str
start_time_utc           str
bidding_zone             str
price                float64
price_unit               str
dtype: object

### Dropping the two columns that carry no information

`price_unit` is constant, so the unit belongs in the column name of the output
(`price_eur_mwh`) rather than in a column of its own. `_id` is an artefact of the
export and has no meaning downstream.

In [8]:
print(prices_raw['price_unit'].value_counts().to_string())
print()
print(prices_raw['bidding_zone'].value_counts().sort_index().to_string())

price_unit
EUR-MWh    194804

bidding_zone
SE1    48542
SE2    48337
SE3    48940
SE4    48985


In [9]:
prices = prices_raw.drop(columns=['_id', 'price_unit'])
prices.head()

,start_time_sweden,start_time_utc,bidding_zone,price
0,2025-12-31T23:45:00,2025-12-31T22:45:00,SE1,15.26
1,2025-12-31T23:45:00,2025-12-31T22:45:00,SE3,30.04
2,2025-12-31T23:45:00,2025-12-31T22:45:00,SE2,20.22
3,2025-12-31T23:45:00,2025-12-31T22:45:00,SE4,33.70
4,2025-12-31T23:30:00,2025-12-31T22:30:00,SE3,33.00


### Parsing the timestamps

Both timestamp columns are read as text by `read_csv`. Parsing them makes the
duplicate check, the hour flooring and the coverage check below possible.

In [10]:
prices['start_time_sweden'] = pd.to_datetime(prices['start_time_sweden'], errors='coerce')
prices['start_time_utc'] = pd.to_datetime(prices['start_time_utc'], errors='coerce')
print('unparseable UTC timestamps: ', int(prices['start_time_utc'].isna().sum()))
print('non-numeric prices:         ', int(pd.to_numeric(prices['price'], errors='coerce').isna().sum()))
prices.dtypes

unparseable UTC timestamps:  0
non-numeric prices:          0


start_time_sweden    datetime64[us]
start_time_utc       datetime64[us]
bidding_zone                    str
price                       float64
dtype: object

### Duplicates, and why the local-time count is misleading

Count duplicates three ways and the numbers disagree. The one that matters is
`(start_time_utc, bidding_zone)`, because that is the real key: one zone cannot
have two different prices for the same instant.

In [11]:
key = ['start_time_utc', 'bidding_zone']
print('duplicates on (start_time_utc, bidding_zone):   ', int(prices.duplicated(subset=key).sum()))
print('duplicates on (start_time_sweden, bidding_zone):', int(prices.duplicated(subset=['start_time_sweden', 'bidding_zone']).sum()))
print('duplicates across every remaining column:       ', int(prices.duplicated().sum()))

duplicates on (start_time_utc, bidding_zone):    3
duplicates on (start_time_sweden, bidding_zone): 35
duplicates across every remaining column:        3


The gap between the two counts is the **daylight saving fall-back hour**. On the
last Sunday of October the clock in Sweden goes back one hour, so 02:00 to 03:00
local happens twice. Those two hours are different instants with different prices,
and in UTC they are correctly distinct. Keying on local time would silently throw
one of them away.

The rows below are duplicated on local time but not on UTC. Note the dates.

In [12]:
dst_rows = prices[
    prices.duplicated(subset=['start_time_sweden', 'bidding_zone'], keep=False)
    & ~prices.duplicated(subset=key, keep=False)
]
print('rows involved:', len(dst_rows))
dst_rows.sort_values(['start_time_sweden', 'bidding_zone']).head(8)

rows involved: 64


,start_time_sweden,start_time_utc,bidding_zone,price
166067,2021-10-31 02:00:00,2021-10-31 00:00:00,SE1,13.67
166068,2021-10-31 02:00:00,2021-10-31 01:00:00,SE1,13.09
166066,2021-10-31 02:00:00,2021-10-31 01:00:00,SE2,13.09
166071,2021-10-31 02:00:00,2021-10-31 00:00:00,SE2,13.67
166069,2021-10-31 02:00:00,2021-10-31 01:00:00,SE3,13.09
166070,2021-10-31 02:00:00,2021-10-31 00:00:00,SE3,13.67
166065,2021-10-31 02:00:00,2021-10-31 00:00:00,SE4,13.67
166072,2021-10-31 02:00:00,2021-10-31 01:00:00,SE4,13.09


In [13]:
# One fall-back date per year. 2021 to 2024 are hourly, so four rows each (one per
# zone). October 2025 is already at 15 minute resolution, so sixteen rows.
dst_rows['start_time_sweden'].dt.date.value_counts().sort_index()

start_time_sweden
2021-10-31     8
2022-10-30     8
2023-10-29     8
2024-10-27     8
2025-10-26    32
Name: count, dtype: int64

The genuine duplicates are the three rows below. They repeat on every column
including `_id`, so they are a straightforward export artefact and the first copy
of each can be kept without losing anything.

In [14]:
prices[prices.duplicated(subset=key, keep=False)].sort_values(key)

,start_time_sweden,start_time_utc,bidding_zone,price
191999,2021-01-30 11:00:00,2021-01-30 10:00:00,SE1,53.24
192002,2021-01-30 11:00:00,2021-01-30 10:00:00,SE1,53.24
159998,2022-01-02 23:00:00,2022-01-02 22:00:00,SE1,32.93
160000,2022-01-02 23:00:00,2022-01-02 22:00:00,SE1,32.93
159999,2022-01-02 23:00:00,2022-01-02 22:00:00,SE3,32.93
160001,2022-01-02 23:00:00,2022-01-02 22:00:00,SE3,32.93


In [15]:
before = len(prices)
prices = prices.drop_duplicates(subset=key, keep='first')
print(f'dropped {before - len(prices)} duplicate rows, {len(prices):,} remain')

dropped 3 duplicate rows, 194,801 remain


### The switch to 15 minute resolution

Up to autumn 2025 every timestamp falls on the hour. After that the feed publishes
four prices per hour. A count of the minute component shows the split, and a
monthly count of the sub-hourly rows shows when it happened.

In [16]:
prices['start_time_utc'].dt.minute.value_counts().sort_index()

start_time_utc
0     171755
15      7701
30      7657
45      7688
Name: count, dtype: int64

In [17]:
sub_hourly = prices[prices['start_time_utc'].dt.minute != 0]
print(f'sub-hourly rows: {len(sub_hourly):,}, first at {sub_hourly["start_time_utc"].min()}')
sub_hourly['start_time_utc'].dt.to_period('M').value_counts().sort_index()

sub-hourly rows: 23,046, first at 2024-10-15 06:45:00


start_time_utc
2024-10       4
2025-10    6263
2025-11    8258
2025-12    8521
Freq: M, Name: count, dtype: int64

The real switch is mid-October 2025. The four rows in October 2024 are strays a
year early, not the start of the new regime.

### Aggregating everything to hourly means

To get one consistent grain, every timestamp is floored to its UTC hour and the
prices in that hour are averaged per zone. Hourly rows pass through unchanged,
since the mean of a single value is that value. The 15 minute rows collapse into
the hour they start in, which is the simple unweighted average of the four
quarters.

In [18]:
prices['ts_utc'] = prices['start_time_utc'].dt.floor('h')
prices_hourly = (
    prices.groupby(['ts_utc', 'bidding_zone'], as_index=False)['price']
    .mean()
    .rename(columns={'bidding_zone': 'zone', 'price': 'price_eur_mwh'})
    .sort_values(['zone', 'ts_utc'])
    .reset_index(drop=True)
)
prices_hourly['price_eur_mwh'] = prices_hourly['price_eur_mwh'].round(4)
print(f'{len(prices):,} rows in, {len(prices_hourly):,} hourly rows out')
prices_hourly.head()

194,801 rows in, 171,959 hourly rows out


,ts_utc,zone,price_eur_mwh
0,2020-12-31 23:00:00,SE1,24.95
1,2021-01-01 00:00:00,SE1,24.35
2,2021-01-01 01:00:00,SE1,23.98
3,2021-01-01 02:00:00,SE1,23.72
4,2021-01-01 03:00:00,SE1,23.73


The file is defined on the Swedish local calendar, so its first rows are midnight
local on 1 January 2021, which is 23:00 UTC on 31 December 2020. Those four rows
sit one hour before the reference grid. They are reported rather than dropped, so
that nothing disappears without being named; trim them later if a strict
2021 to 2025 UTC window is wanted.

In [19]:
outside = prices_hourly[
    (prices_hourly['ts_utc'] < HOURLY_GRID[0]) | (prices_hourly['ts_utc'] > HOURLY_GRID[-1])
]
print('rows outside the reference grid, kept:', len(outside))
outside

rows outside the reference grid, kept: 4


,ts_utc,zone,price_eur_mwh
0,2020-12-31 23:00:00,SE1,24.95
42922,2020-12-31 23:00:00,SE2,24.95
85825,2020-12-31 23:00:00,SE3,24.95
128892,2020-12-31 23:00:00,SE4,24.95


In [20]:
prices_hourly.to_csv(CLEAN_DIR / 'clean_prices_hourly.csv', index=False)
print('wrote data/clean/clean_prices_hourly.csv', prices_hourly.shape)
prices_hourly.head()

wrote data/clean/clean_prices_hourly.csv (171959, 3)


,ts_utc,zone,price_eur_mwh
0,2020-12-31 23:00:00,SE1,24.95
1,2021-01-01 00:00:00,SE1,24.35
2,2021-01-01 01:00:00,SE1,23.98
3,2021-01-01 02:00:00,SE1,23.72
4,2021-01-01 03:00:00,SE1,23.73


---
# 2. SCB production and usage

Monthly electricity production and usage in GWh, by category and bidding zone,
downloaded from Statistics Sweden. Two problems:

1. **It is a spreadsheet export, not a data file.** The first line is a title and
   the second is blank, so the real header is on the third line.
2. **It is wide.** Each month is its own column, 66 of them, running six months
   past the period this project covers.

In [21]:
scb_file = next(f for f in raw_files if 'scb' in f.name.lower())
for line in open(scb_file, encoding='utf-8-sig').read().splitlines()[:3]:
    print(repr(line[:100]))

'"Electricity production, net and usage, in GWh by Production and usage, bidding zone and month"'
''
'"Production and usage","bidding zone","2021M01","2021M02","2021M03","2021M04","2021M05","2021M06","2'


In [22]:
# Skip the title line and the blank line; the third line is the header.
scb_raw = pd.read_csv(scb_file, skiprows=2, encoding='utf-8-sig').dropna(how='all')
print(scb_raw.shape)
scb_raw.iloc[:5, :7]

(48, 68)


,Production and usage,bidding zone,2021M01,2021M02,2021M03,2021M04,2021M05
0,total production,SE1,2690,2491,2460,2228,2047
1,total production,SE2,5156,4786,4958,4629,4299
2,total production,SE3,7868,7400,7761,7287,5835
3,total production,SE4,842,801,805,764,521
4,"hydro power (including pump power), net",SE1,2320,1973,1761,1655,1665


In [23]:
id_cols = ['Production and usage', 'bidding zone']
month_cols = [c for c in scb_raw.columns if len(str(c)) == 7 and str(c)[4] == 'M']
print(f'{len(month_cols)} month columns, {month_cols[0]} to {month_cols[-1]}')
print('columns that are neither id nor month:', [c for c in scb_raw.columns if c not in id_cols + month_cols])
print()
print(f'{scb_raw[id_cols[0]].nunique()} categories x {scb_raw[id_cols[1]].nunique()} zones = {len(scb_raw)} rows')
scb_raw[id_cols[0]].unique()

66 month columns, 2021M01 to 2026M06
columns that are neither id nor month: []

12 categories x 4 zones = 48 rows


<ArrowStringArray>
[                          'total production',    'hydro power (including pump power), net',                                 'wind power',
                      'solar power (on-grid)',            'nuclear power (condensing), net',            'conventional thermal power, net',
                                'total usage',                   'mining and manufacturing',    'electricity, gas, heat and water plants',
            'railways, trams and bus traffic', 'other (residential sector, services, etc.)',                                     'losses']
Length: 12, dtype: str

### Wide to long

One row per category, zone and month. 48 rows times 66 months gives 3,168 rows
before the period is trimmed.

In [24]:
scb_long = scb_raw.melt(
    id_vars=id_cols,
    value_vars=month_cols,
    var_name='month_str',
    value_name='value_gwh',
).rename(columns={'Production and usage': 'category', 'bidding zone': 'zone'})
print(f'{len(scb_raw)} rows x {len(month_cols)} months = {len(scb_long):,} long rows')
scb_long.head()

48 rows x 66 months = 3,168 long rows


,category,zone,month_str,value_gwh
0,total production,SE1,2021M01,2690
1,total production,SE2,2021M01,5156
2,total production,SE3,2021M01,7868
3,total production,SE4,2021M01,842
4,"hydro power (including pump power), net",SE1,2021M01,2320


`2021M01` is SCB's month notation. Turning it into a real date, the first of the
month, makes the column sortable and joinable.

In [25]:
scb_long['month'] = pd.to_datetime(
    scb_long['month_str'].str.replace('M', '-', regex=False), format='%Y-%m'
)
scb_long[['month_str', 'month']].drop_duplicates().head()

,month_str,month
0,2021M01,2021-01-01
48,2021M02,2021-02-01
96,2021M03,2021-03-01
144,2021M04,2021-04-01
192,2021M05,2021-05-01


In [26]:
scb_long['value_gwh'] = pd.to_numeric(scb_long['value_gwh'], errors='coerce')
print('blank or non-numeric values:', int(scb_long['value_gwh'].isna().sum()))
scb_long.dtypes

blank or non-numeric values: 0


category                str
zone                    str
month_str               str
value_gwh             int64
month        datetime64[us]
dtype: object

### Trimming to 2021M01 through 2025M12

The download runs to 2026M06, six months past the period the other two datasets
cover. Those months are dropped, and the months dropped are printed so the cut is
visible rather than implied.

In [27]:
dropped_months = sorted(scb_long.loc[scb_long['month'] > MONTHLY_GRID[-1], 'month_str'].unique())
scb_clean = scb_long[
    (scb_long['month'] >= MONTHLY_GRID[0]) & (scb_long['month'] <= MONTHLY_GRID[-1])
]
scb_clean = (
    scb_clean[['category', 'zone', 'month', 'value_gwh']]
    .sort_values(['category', 'zone', 'month'])
    .reset_index(drop=True)
)
print('dropped months:', dropped_months)
print(f'{len(scb_long):,} rows in, {len(scb_clean):,} rows out (48 x 60 = 2,880 expected)')

dropped months: ['2026M01', '2026M02', '2026M03', '2026M04', '2026M05', '2026M06']
3,168 rows in, 2,880 rows out (48 x 60 = 2,880 expected)


In [28]:
# Every category and zone should have all 60 months.
scb_counts = scb_clean.groupby(['category', 'zone']).size()
print(f'{len(scb_counts)} category/zone series, each expecting {len(MONTHLY_GRID)} months')
print('series with a missing month:', int((scb_counts != len(MONTHLY_GRID)).sum()))
scb_counts.head()

48 category/zone series, each expecting 60 months
series with a missing month: 0


category                                 zone
conventional thermal power, net          SE1     60
                                         SE2     60
                                         SE3     60
                                         SE4     60
electricity, gas, heat and water plants  SE1     60
dtype: int64

In [29]:
scb_clean.to_csv(CLEAN_DIR / 'clean_scb_monthly.csv', index=False)
print('wrote data/clean/clean_scb_monthly.csv', scb_clean.shape)
scb_clean.head()

wrote data/clean/clean_scb_monthly.csv (2880, 4)


,category,zone,month,value_gwh
0,"conventional thermal power, net",SE1,2021-01-01,157
1,"conventional thermal power, net",SE1,2021-02-01,134
2,"conventional thermal power, net",SE1,2021-03-01,136
3,"conventional thermal power, net",SE1,2021-04-01,124
4,"conventional thermal power, net",SE1,2021-05-01,114


---
# 3. SMHI hourly temperature

Four separate downloads, one per zone reference station. Each file is semicolon
separated, encoded as UTF-8 with a byte order mark, and opens with a block of
station metadata: station name and number, the parameter description, the period
of record. Only after that block does the real header appear.

**The metadata block is not the same length in every file.** Hardcoding a skip
count would work for three of the four and quietly mangle the fourth, so the header
row is located by its content instead.

In [30]:
smhi_files = sorted(f for f in raw_files if f.name[:2] == 'SE' and f.name[2] in '1234')
smhi_files

[WindowsPath('C:/Users/ewuzi/Downloads/Project/data/raw/SE1 Luleå-Kallax.csv'),
 WindowsPath('C:/Users/ewuzi/Downloads/Project/data/raw/SE2 Östersund-Frösön.csv'),
 WindowsPath('C:/Users/ewuzi/Downloads/Project/data/raw/SE3 Stockholm-Arlanda.csv'),
 WindowsPath('C:/Users/ewuzi/Downloads/Project/data/raw/SE4 Helsingborg A.csv')]

In [31]:
# The top of one file: metadata first, then the header row.
for i, line in enumerate(open(smhi_files[0], encoding='utf-8-sig').read().splitlines()[:12]):
    print(f'{i:>2}  {line[:85]}')

 0  ﻿Stationsnamn;Stationsnummer;Stationsnät;Mäthöjd (meter över marken)
 1  Luleå-Kallax Flygplats;162860;SMHIs stationsnät;2.0
 2  
 3  Parameternamn;Beskrivning;Enhet
 4  Lufttemperatur;momentanvärde, 1 gång/tim;celsius
 5  
 6  Tidsperiod (fr.o.m);Tidsperiod (t.o.m);Höjd (meter över havet);Latitud (decimalgrader
 7  1945-01-01 00:00:00;2026-09-01 06:20:15;19.9;65.5430;22.1240
 8  
 9  Datum;Tid (UTC);Lufttemperatur;Kvalitet;;Tidsutsnitt:
10  2021-01-01;06:00:00;-1.5;G
11  2021-01-01;12:00:00;-2.5;G


In [32]:
# Find the header row by its content, per file. Note the differing line numbers.
HEADER_START = 'Datum;Tid (UTC)'
header_rows = {}
for f in smhi_files:
    with open(f, encoding='utf-8-sig') as fh:
        header_rows[f] = next(i for i, line in enumerate(fh) if line.startswith(HEADER_START))
header_rows

{WindowsPath('C:/Users/ewuzi/Downloads/Project/data/raw/SE1 Luleå-Kallax.csv'): 9,
 WindowsPath('C:/Users/ewuzi/Downloads/Project/data/raw/SE2 Östersund-Frösön.csv'): 10,
 WindowsPath('C:/Users/ewuzi/Downloads/Project/data/raw/SE3 Stockholm-Arlanda.csv'): 9,
 WindowsPath('C:/Users/ewuzi/Downloads/Project/data/raw/SE4 Helsingborg A.csv'): 9}

The header line ends with two extra fields (an empty one and `Tidsutsnitt:`) that
the data rows do not use, so only the first four columns are read. The zone comes
from the filename prefix, which is how the downloads were named.

In [33]:
frames = []
for f in smhi_files:
    part = pd.read_csv(
        f,
        sep=';',
        skiprows=header_rows[f],
        header=0,
        usecols=[0, 1, 2, 3],
        names=['date', 'time', 'temp_c', 'quality'],
        encoding='utf-8-sig',
        dtype=str,
    )
    part['zone'] = f.name[:3]
    print(f'{f.name[:3]}  header line {header_rows[f]:>2}  {len(part):,} data rows  {f}')
    frames.append(part)

temp_raw = pd.concat(frames, ignore_index=True)
print()
print('concatenated:', temp_raw.shape)
temp_raw.head()

SE1  header line  9  43,643 data rows  C:\Users\ewuzi\Downloads\Project\data\raw\SE1 Luleå-Kallax.csv


SE2  header line 10  43,649 data rows  C:\Users\ewuzi\Downloads\Project\data\raw\SE2 Östersund-Frösön.csv


SE3  header line  9  43,707 data rows  C:\Users\ewuzi\Downloads\Project\data\raw\SE3 Stockholm-Arlanda.csv


SE4  header line  9  43,814 data rows  C:\Users\ewuzi\Downloads\Project\data\raw\SE4 Helsingborg A.csv



concatenated: (174813, 5)


,date,time,temp_c,quality,zone
0,2021-01-01,06:00:00,-1.5,G,SE1
1,2021-01-01,12:00:00,-2.5,G,SE1
2,2021-01-01,18:00:00,-3.6,G,SE1
3,2021-01-01,19:00:00,-3.7,G,SE1
4,2021-01-01,20:00:00,-3.8,G,SE1


### Building the timestamp

Date and time sit in separate columns and the time is already UTC, so the two are
simply joined. Nothing is shifted or localised.

In [34]:
temp_raw['ts_utc'] = pd.to_datetime(
    temp_raw['date'].str.strip() + ' ' + temp_raw['time'].str.strip(),
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce',
)
temp_raw['temp_c'] = pd.to_numeric(temp_raw['temp_c'], errors='coerce')
temp_raw['quality'] = temp_raw['quality'].str.strip()
print('unparseable timestamps:  ', int(temp_raw['ts_utc'].isna().sum()))
print('non-numeric temperatures:', int(temp_raw['temp_c'].isna().sum()))
print('duplicate (zone, ts_utc):', int(temp_raw.duplicated(subset=['zone', 'ts_utc']).sum()))
temp_raw.head()

unparseable timestamps:   0
non-numeric temperatures: 0
duplicate (zone, ts_utc): 0


,date,time,temp_c,quality,zone,ts_utc
0,2021-01-01,06:00:00,-1.5,G,SE1,2021-01-01 06:00:00
1,2021-01-01,12:00:00,-2.5,G,SE1,2021-01-01 12:00:00
2,2021-01-01,18:00:00,-3.6,G,SE1,2021-01-01 18:00:00
3,2021-01-01,19:00:00,-3.7,G,SE1,2021-01-01 19:00:00
4,2021-01-01,20:00:00,-3.8,G,SE1,2021-01-01 20:00:00


`Kvalitet` is SMHI's quality flag: `G` for checked and approved, `Y` for suspect
but plausible. Both are kept and the flag is carried into the output, so that a
later analysis can decide for itself whether to exclude the `Y` readings. Dropping
them here would hide a choice inside a cleaning step.

In [35]:
temp_raw['quality'].value_counts(dropna=False)

quality
G    174780
Y        33
Name: count, dtype: int64

In [36]:
temperature = temp_raw.dropna(subset=['ts_utc']).drop_duplicates(subset=['zone', 'ts_utc'], keep='first')
temperature = (
    temperature[['ts_utc', 'zone', 'temp_c', 'quality']]
    .sort_values(['zone', 'ts_utc'])
    .reset_index(drop=True)
)
print(f'{len(temp_raw):,} rows in, {len(temperature):,} rows out')
temperature.head()

174,813 rows in, 174,813 rows out


,ts_utc,zone,temp_c,quality
0,2021-01-01 06:00:00,SE1,-1.5,G
1,2021-01-01 12:00:00,SE1,-2.5,G
2,2021-01-01 18:00:00,SE1,-3.6,G
3,2021-01-01 19:00:00,SE1,-3.7,G
4,2021-01-01 20:00:00,SE1,-3.8,G


In [37]:
temperature.to_csv(CLEAN_DIR / 'clean_temperature_hourly.csv', index=False)
print('wrote data/clean/clean_temperature_hourly.csv', temperature.shape)

wrote data/clean/clean_temperature_hourly.csv (174813, 4)


---
# 4. Cross-border physical flows

Physical electricity flows between Sweden and its neighbours, one file per year,
`data/raw/flows_se_2021.json` to `data/raw/flows_se_2025.json`. Source: the Energy-Charts API run by
Fraunhofer ISE, licensed CC BY 4.0.

Each file holds a `unix_seconds` array and a `countries` array. Every entry in
`countries` has a `name` and a `data` array the same length as `unix_seconds`, so
the file is a bundle of parallel arrays rather than a table. The partners are
Denmark, Finland, Germany, Lithuania, Norway and Poland, plus a `sum` series that
adds them together.

Values are in **GW**, signed from Sweden's point of view: **negative is export from
Sweden, positive is import**.

Three things to watch:

1. **Parallel arrays, not rows.** Each file has to be reshaped before it is a table.
2. **The resolution changes in 2025.** The 2021 to 2024 files are hourly, 8,760 rows
   a year and 8,784 in the leap year 2024. The 2025 file has 35,040 rows, which is
   15 minute resolution: the same market change that shows up in the price data.
3. **These are national flows, not per bidding zone.** There is one series per
   partner country for Sweden as a whole, so this table has no `zone` column and
   cannot be split across SE1 to SE4.

In [38]:
import json

flow_files = sorted(RAW_DIR.glob('flows_se_*.json'))
for f in flow_files:
    print(f'{os.path.getsize(f) / 1e6:6.2f} MB   {f}')

  0.50 MB   C:\Users\ewuzi\Downloads\Project\data\raw\flows_se_2021.json
  0.51 MB   C:\Users\ewuzi\Downloads\Project\data\raw\flows_se_2022.json
  0.50 MB   C:\Users\ewuzi\Downloads\Project\data\raw\flows_se_2023.json
  0.50 MB   C:\Users\ewuzi\Downloads\Project\data\raw\flows_se_2024.json
  2.02 MB   C:\Users\ewuzi\Downloads\Project\data\raw\flows_se_2025.json


In [39]:
# What one file actually contains.
sample = json.load(open(flow_files[0], encoding='utf-8'))
print('top-level keys:', list(sample.keys()))
print('timestamps:', f"{len(sample['unix_seconds']):,}", '| first value:', sample['unix_seconds'][0])
print('series:', [(c['name'], len(c['data'])) for c in sample['countries']])

top-level keys: ['unix_seconds', 'countries', 'deprecated']
timestamps: 8,760 | first value: 1609455600
series: [('Denmark', 8760), ('Finland', 8760), ('Germany', 8760), ('Lithuania', 8760), ('Norway', 8760), ('Poland', 8760), ('sum', 8760)]


The first timestamp is 23:00 UTC on 31 December 2020, which is midnight local on
1 January 2021. Like the price file, these downloads are cut on the Swedish local
calendar rather than the UTC one.

Reshaping a file means pairing `unix_seconds` with each country's `data` array.
`unit='s'` tells pandas the integers are seconds since the epoch; `utc=True` reads
them as UTC instead of guessing a local zone, and `tz_localize(None)` then drops the
timezone label so these timestamps match the naive UTC ones used everywhere else in
the notebook.

In [40]:
def read_flow_file(path):
    """One Energy-Charts JSON file into a table: one row per timestamp, one column per partner."""
    with open(path, encoding='utf-8') as fh:
        raw = json.load(fh)
    frame = pd.DataFrame({c['name']: c['data'] for c in raw['countries']})
    stamps = pd.to_datetime(raw['unix_seconds'], unit='s', utc=True).tz_localize(None)
    frame.insert(0, 'ts_utc', stamps)
    return frame

In [41]:
flows_2021 = read_flow_file(flow_files[0])
print(flows_2021.shape)
flows_2021.head()

(8760, 8)


,ts_utc,Denmark,Finland,Germany,Lithuania,Norway,Poland,sum
0,2020-12-31 23:00:00,-1.769,-1.881,-0.606,-0.278,0.984,-0.546,-4.096
1,2021-01-01 00:00:00,-1.770,-1.434,-0.607,-0.490,0.533,-0.328,-4.096
2,2021-01-01 01:00:00,-1.763,-1.314,-0.606,-0.429,-0.107,-0.009,-4.228
3,2021-01-01 02:00:00,-1.758,-1.232,-0.606,-0.388,-0.348,0.000,-4.332
4,2021-01-01 03:00:00,-1.713,-1.292,-0.607,-0.394,-0.250,0.000,-4.256


### Resampling every year to hourly means

The 2025 file is four times as dense as the others, so the years cannot simply be
stacked. `resample('h').mean()` puts every file on the same grid and labels each
bin with the hour it starts in, which matches how the price data was floored in
section 1. A file that is already hourly comes back unchanged, since the mean of a
single value is that value, so the same call works for all five years without a
special case.

In [42]:
rows_report = []
hourly_frames = []
for f in flow_files:
    year_frame = read_flow_file(f)
    hourly = year_frame.set_index('ts_utc').resample('h').mean().reset_index()
    rows_report.append({
        'file': f,
        'raw_rows': len(year_frame),
        'hourly_rows': len(hourly),
        'first': str(year_frame['ts_utc'].min()),
        'last': str(year_frame['ts_utc'].max()),
    })
    hourly_frames.append(hourly)

pd.DataFrame(rows_report)

,file,raw_rows,hourly_rows,first,last
0,C:\Users\ewuzi\Downloads\Project\data\raw\flow...,8760,8760,2020-12-31 23:00:00,2021-12-31 22:00:00
1,C:\Users\ewuzi\Downloads\Project\data\raw\flow...,8760,8760,2021-12-31 23:00:00,2022-12-31 22:00:00
2,C:\Users\ewuzi\Downloads\Project\data\raw\flow...,8760,8760,2022-12-31 23:00:00,2023-12-31 22:00:00
3,C:\Users\ewuzi\Downloads\Project\data\raw\flow...,8784,8784,2023-12-31 23:00:00,2024-12-31 22:00:00
4,C:\Users\ewuzi\Downloads\Project\data\raw\flow...,35040,8760,2024-12-31 23:00:00,2025-12-31 22:45:00


In [43]:
flows = pd.concat(hourly_frames, ignore_index=True).sort_values('ts_utc').reset_index(drop=True)
print('concatenated:', flows.shape)
flows.head()

concatenated: (43824, 8)


,ts_utc,Denmark,Finland,Germany,Lithuania,Norway,Poland,sum
0,2020-12-31 23:00:00,-1.769,-1.881,-0.606,-0.278,0.984,-0.546,-4.096
1,2021-01-01 00:00:00,-1.770,-1.434,-0.607,-0.490,0.533,-0.328,-4.096
2,2021-01-01 01:00:00,-1.763,-1.314,-0.606,-0.429,-0.107,-0.009,-4.228
3,2021-01-01 02:00:00,-1.758,-1.232,-0.606,-0.388,-0.348,0.000,-4.332
4,2021-01-01 03:00:00,-1.713,-1.292,-0.607,-0.394,-0.250,0.000,-4.256


In [44]:
# The files are cut on the local calendar, so the year boundaries could overlap by
# an hour. Check rather than assume.
print('duplicate timestamps:', int(flows.duplicated(subset=['ts_utc']).sum()))
print('gaps between consecutive rows:', flows['ts_utc'].diff().value_counts().to_dict())

duplicate timestamps: 0
gaps between consecutive rows: {Timedelta('0 days 01:00:00'): 43823}


### Naming and the sign convention

`sum` is renamed to `sum_gw` so the unit is visible in the column name and the word
does not shadow the Python builtin when the file is read back.

In [45]:
flows.columns = [c.lower() for c in flows.columns]
flows = flows.rename(columns={'sum': 'sum_gw'})
value_cols = ['denmark', 'finland', 'germany', 'lithuania', 'norway', 'poland', 'sum_gw']
flows = flows[['ts_utc'] + value_cols]
flows[value_cols] = flows[value_cols].round(4)
print(flows.dtypes.to_string())
flows.head()

ts_utc       datetime64[s]
denmark            float64
finland            float64
germany            float64
lithuania          float64
norway             float64
poland             float64
sum_gw             float64


,ts_utc,denmark,finland,germany,lithuania,norway,poland,sum_gw
0,2020-12-31 23:00:00,-1.769,-1.881,-0.606,-0.278,0.984,-0.546,-4.096
1,2021-01-01 00:00:00,-1.770,-1.434,-0.607,-0.490,0.533,-0.328,-4.096
2,2021-01-01 01:00:00,-1.763,-1.314,-0.606,-0.429,-0.107,-0.009,-4.228
3,2021-01-01 02:00:00,-1.758,-1.232,-0.606,-0.388,-0.348,0.000,-4.332
4,2021-01-01 03:00:00,-1.713,-1.292,-0.607,-0.394,-0.250,0.000,-4.256


In [46]:
# Sanity check on the sign convention: Sweden is a net exporter most of the time,
# so sum_gw should be negative in the majority of hours.
net_export_share = (flows['sum_gw'] < 0).mean()
print(f'hours with net export (sum_gw < 0): {net_export_share:.1%}')
print('missing values per column:')
print(flows[value_cols].isna().sum().to_string())
flows[value_cols].describe().round(2)

hours with net export (sum_gw < 0): 97.9%
missing values per column:
denmark      0
finland      0
germany      0
lithuania    0
norway       0
poland       0
sum_gw       0


,denmark,finland,germany,lithuania,norway,poland,sum_gw
count,43824.00,43824.00,43824.00,43824.00,43824.00,43824.00,43824.00
mean,-0.90,-1.25,-0.30,-0.51,-0.19,-0.38,-3.51
std,0.93,0.95,0.33,0.29,1.37,0.31,1.61
min,-2.11,-3.14,-0.72,-0.74,-4.35,-0.60,-9.04
25%,-1.62,-2.01,-0.61,-0.73,-1.15,-0.60,-4.63
50%,-1.13,-1.35,-0.38,-0.66,-0.16,-0.58,-3.55
75%,-0.42,-0.68,0.00,-0.36,0.79,-0.21,-2.46
max,2.50,2.26,0.60,0.70,3.64,0.64,3.30


In [47]:
flow_gaps = HOURLY_GRID.difference(pd.DatetimeIndex(flows['ts_utc']))
outside_flows = flows[(flows['ts_utc'] < HOURLY_GRID[0]) | (flows['ts_utc'] > HOURLY_GRID[-1])]
print(f'rows: {len(flows):,} (43,824 expected)')
print(f'missing hours against the {len(HOURLY_GRID):,} hour grid: {len(flow_gaps)}  {list(flow_gaps)}')
print(f'rows outside the grid, kept: {len(outside_flows)}')
flows['ts_utc'].dt.year.value_counts().sort_index()

rows: 43,824 (43,824 expected)
missing hours against the 43,824 hour grid: 1  [Timestamp('2025-12-31 23:00:00')]
rows outside the grid, kept: 1


ts_utc
2020       1
2021    8760
2022    8760
2023    8760
2024    8784
2025    8759
Name: count, dtype: int64

The one missing hour and the one extra row are the same local-calendar offset seen
in the price data: the series runs from 23:00 UTC on 31 December 2020 to 22:00 UTC
on 31 December 2025. Nothing is dropped to hide it.

In [48]:
flows.to_csv(CLEAN_DIR / 'clean_flows_hourly.csv', index=False)
print('wrote data/clean/clean_flows_hourly.csv', flows.shape)
flows.head()

wrote data/clean/clean_flows_hourly.csv (43824, 8)


,ts_utc,denmark,finland,germany,lithuania,norway,poland,sum_gw
0,2020-12-31 23:00:00,-1.769,-1.881,-0.606,-0.278,0.984,-0.546,-4.096
1,2021-01-01 00:00:00,-1.770,-1.434,-0.607,-0.490,0.533,-0.328,-4.096
2,2021-01-01 01:00:00,-1.763,-1.314,-0.606,-0.429,-0.107,-0.009,-4.228
3,2021-01-01 02:00:00,-1.758,-1.232,-0.606,-0.388,-0.348,0.000,-4.332
4,2021-01-01 03:00:00,-1.713,-1.292,-0.607,-0.394,-0.250,0.000,-4.256


---
# 5. National generation and load

Hourly electricity generation by source, plus load, for Sweden as a whole. Five
files, `data/raw/generation_se_2021.json` to `data/raw/generation_se_2025.json`, from the same
Energy-Charts API as the flow data (Fraunhofer ISE, CC BY 4.0).

The shape is the same bundle of parallel arrays as section 4, with one difference:
the series sit under `production_types` rather than `countries`. Values are in
**MW**, except the two `Renewable share` series, which are percentages.

**These are national figures, not per bidding zone.** Energy-Charts does not serve
Swedish generation at bidding-zone level. Passing the `bzn` parameter does not
fail: it is silently ignored and the response comes back as Germany, which is worse
than an error, because nothing in the output says so. That is why the hourly
generation work in this project is national, and why anything that has to be split
across SE1 to SE4 stays monthly and comes from the SCB file in section 2.

Three problems to handle:

1. **Two series start late.** Solar and Fossil gas are null for the whole of 2021
   until 14 December. Energy-Charts did not report them for Sweden before then.
2. **A handful of single-hour holes.** Scattered nulls in 2022 and 2024, one hour
   at a time, each with valid values on either side.
3. **No resampling needed, but worth verifying.** Unlike the price and flow data,
   all five files should already be hourly. That is checked below rather than
   assumed.

In [49]:
gen_files = sorted(RAW_DIR.glob('generation_se_*.json'))
for f in gen_files:
    print(f'{os.path.getsize(f) / 1e6:6.2f} MB   {f}')

  0.72 MB   C:\Users\ewuzi\Downloads\Project\data\raw\generation_se_2021.json
  0.71 MB   C:\Users\ewuzi\Downloads\Project\data\raw\generation_se_2022.json
  0.71 MB   C:\Users\ewuzi\Downloads\Project\data\raw\generation_se_2023.json
  0.71 MB   C:\Users\ewuzi\Downloads\Project\data\raw\generation_se_2024.json
  0.71 MB   C:\Users\ewuzi\Downloads\Project\data\raw\generation_se_2025.json


In [50]:
# What one file contains, and where the nulls are.
gen_sample = json.load(open(gen_files[0], encoding='utf-8'))
print('top-level keys:', list(gen_sample.keys()))
print('timestamps:', f"{len(gen_sample['unix_seconds']):,}")
for t in gen_sample['production_types']:
    nulls = sum(1 for v in t['data'] if v is None)
    print(f"  {t['name']:<32} {len(t['data']):>6,} values, {nulls:>5,} null")

top-level keys: ['unix_seconds', 'production_types', 'deprecated']
timestamps: 8,760
  Cross border electricity trading  8,760 values,     0 null
  Nuclear                           8,760 values,     0 null
  Hydro water reservoir             8,760 values,     0 null
  Others                            8,760 values,     0 null
  Wind onshore                      8,760 values,     0 null
  Load                              8,760 values,     0 null
  Residual load                     8,760 values,     0 null
  Renewable share of load           8,760 values,     0 null
  Renewable share of generation     8,760 values,     0 null
  Fossil gas                        8,760 values, 8,352 null
  Solar                             8,760 values, 8,352 null


In [51]:
def read_generation_file(path):
    """One Energy-Charts generation file into a table: one row per timestamp, one column per series."""
    with open(path, encoding='utf-8') as fh:
        raw = json.load(fh)
    frame = pd.DataFrame({t['name']: t['data'] for t in raw['production_types']})
    stamps = pd.to_datetime(raw['unix_seconds'], unit='s', utc=True).tz_localize(None)
    frame.insert(0, 'ts_utc', stamps)
    return frame

In [52]:
gen_2021 = read_generation_file(gen_files[0])
print(gen_2021.shape)
gen_2021.head()

(8760, 12)


,ts_utc,Cross border electricity trading,Nuclear,Hydro water reservoir,Others,Wind onshore,Load,Residual load,Renewable share of load,Renewable share of generation,Fossil gas,Solar
0,2020-12-31 23:00:00,-3979.5,6771.0,10552.0,1044.0,1030.0,15678.0,14648.0,73.9,59.7,NaN,NaN
1,2021-01-01 00:00:00,-4143.7,6773.0,10419.0,1038.0,978.0,15507.0,14529.0,73.5,59.3,NaN,NaN
2,2021-01-01 01:00:00,-4355.9,6772.0,10280.0,1010.0,913.0,15130.0,14217.0,74.0,59.0,NaN,NaN
3,2021-01-01 02:00:00,-4389.4,6775.0,10221.0,997.0,914.0,14917.0,14003.0,74.6,58.9,NaN,NaN
4,2021-01-01 03:00:00,-4519.3,6775.0,10302.0,998.0,959.0,15029.0,14070.0,74.9,59.2,NaN,NaN


### Checking that the files really are hourly

The claim is that no resampling is needed. That is easy to verify: every gap between
consecutive timestamps should be exactly one hour, so each file should report a
single distinct gap, and the row counts should be 8,760 with 8,784 for the leap year
2024.

In [53]:
gen_check = []
gen_frames = []
for f in gen_files:
    year_frame = read_generation_file(f)
    gaps = year_frame['ts_utc'].diff().dropna().unique()
    gen_check.append({
        'file': f,
        'rows': len(year_frame),
        'distinct_gaps': [str(x) for x in gaps],
        'first': str(year_frame['ts_utc'].min()),
        'last': str(year_frame['ts_utc'].max()),
    })
    gen_frames.append(year_frame)

pd.DataFrame(gen_check)

,file,rows,distinct_gaps,first,last
0,C:\Users\ewuzi\Downloads\Project\data\raw\gene...,8760,[0 days 01:00:00],2020-12-31 23:00:00,2021-12-31 22:00:00
1,C:\Users\ewuzi\Downloads\Project\data\raw\gene...,8760,[0 days 01:00:00],2021-12-31 23:00:00,2022-12-31 22:00:00
2,C:\Users\ewuzi\Downloads\Project\data\raw\gene...,8760,[0 days 01:00:00],2022-12-31 23:00:00,2023-12-31 22:00:00
3,C:\Users\ewuzi\Downloads\Project\data\raw\gene...,8784,[0 days 01:00:00],2023-12-31 23:00:00,2024-12-31 22:00:00
4,C:\Users\ewuzi\Downloads\Project\data\raw\gene...,8760,[0 days 01:00:00],2024-12-31 23:00:00,2025-12-31 22:00:00


In [54]:
generation = pd.concat(gen_frames, ignore_index=True).sort_values('ts_utc').reset_index(drop=True)
print('concatenated:', generation.shape, ' (43,824 expected)')
print('duplicate timestamps at the year boundaries:', int(generation.duplicated(subset=['ts_utc']).sum()))
generation.head()

concatenated: (43824, 12)  (43,824 expected)
duplicate timestamps at the year boundaries: 0


,ts_utc,Cross border electricity trading,Nuclear,Hydro water reservoir,Others,Wind onshore,Load,Residual load,Renewable share of load,Renewable share of generation,Fossil gas,Solar
0,2020-12-31 23:00:00,-3979.5,6771.0,10552.0,1044.0,1030.0,15678.0,14648.0,73.9,59.7,NaN,NaN
1,2021-01-01 00:00:00,-4143.7,6773.0,10419.0,1038.0,978.0,15507.0,14529.0,73.5,59.3,NaN,NaN
2,2021-01-01 01:00:00,-4355.9,6772.0,10280.0,1010.0,913.0,15130.0,14217.0,74.0,59.0,NaN,NaN
3,2021-01-01 02:00:00,-4389.4,6775.0,10221.0,997.0,914.0,14917.0,14003.0,74.6,58.9,NaN,NaN
4,2021-01-01 03:00:00,-4519.3,6775.0,10302.0,998.0,959.0,15029.0,14070.0,74.9,59.2,NaN,NaN


### The missing values

Two different problems hide in the nulls and they need different treatment, so they
are counted separately before anything is filled.

In [55]:
null_by_column = generation.drop(columns=['ts_utc']).isna().sum()
print('nulls per series:')
print(null_by_column[null_by_column > 0].to_string())
print()
nulls_by_year = generation.drop(columns=['ts_utc']).isna().groupby(generation['ts_utc'].dt.year).sum()
print('nulls per year, series with at least one:')
nulls_by_year.loc[:, nulls_by_year.sum() > 0]

nulls per series:
Nuclear                             1
Hydro water reservoir               2
Others                              1
Wind onshore                        1
Renewable share of load             1
Renewable share of generation       1
Fossil gas                       8353
Solar                            8353

nulls per year, series with at least one:


,Nuclear,Hydro water reservoir,Others,Wind onshore,Renewable share of load,Renewable share of generation,Fossil gas,Solar
ts_utc,,,,,,,,
2020,0,0,0,0,0,0,1,1
2021,0,0,0,0,0,0,8351,8351
2022,1,1,1,1,1,1,1,1
2023,0,0,0,0,0,0,0,0
2024,0,1,0,0,0,0,0,0
2025,0,0,0,0,0,0,0,0


In [56]:
late_start = ['Solar', 'Fossil gas']
first_seen = {c: generation.loc[generation[c].notna(), 'ts_utc'].min() for c in late_start}
for c, t in first_seen.items():
    before = int(generation.loc[generation['ts_utc'] < t, c].isna().sum())
    print(f'{c:<12} first non-null at {t}, {before:,} null hours before it')

Solar        first non-null at 2021-12-14 23:00:00, 8,352 null hours before it
Fossil gas   first non-null at 2021-12-14 23:00:00, 8,352 null hours before it


### Solar and gas in 2021: an absence of reporting, not an absence of power

Those 8,352 hours each are not a gap in Swedish electricity, they are a gap in what
Energy-Charts published. Sweden had very little solar capacity in 2021 and gas-fired
generation was marginal, a few hundred MW at most in a system whose load reaches
27,000 MW. Setting the window to zero is therefore close to the truth for the
variables of interest here, and it keeps the whole of 2021 usable. The alternative,
dropping the year, would cost a fifth of the sample to avoid an error of a fraction
of a percent.

The assumption is applied only to those two series, and only before their first
reported value, so it stays visible instead of being buried in a `fillna(0)` across
the table. Anyone who needs exact 2021 solar or gas figures should treat that window
as unreported rather than as zero.

In [57]:
zero_filled = 0
for c in late_start:
    mask = (generation['ts_utc'] < first_seen[c]) & generation[c].isna()
    zero_filled += int(mask.sum())
    generation.loc[mask, c] = 0.0

print(f'set to zero: {zero_filled:,} values, across {late_start}')
remaining = generation.drop(columns=['ts_utc']).isna().sum()
print('nulls still to deal with:')
print(remaining[remaining > 0].to_string())

set to zero: 16,704 values, across ['Solar', 'Fossil gas']
nulls still to deal with:
Nuclear                          1
Hydro water reservoir            2
Others                           1
Wind onshore                     1
Renewable share of load          1
Renewable share of generation    1
Fossil gas                       1
Solar                            1


### The single-hour holes

What is left is a small number of isolated hours in 2022 and 2024. Each has a valid
value on either side, so a linear interpolation is the least invasive fix and at one
hour wide it cannot drift far. The rows are shown first, then the fill is counted.

In [58]:
value_names = [c for c in generation.columns if c != 'ts_utc']
gap_rows = generation[generation[value_names].isna().any(axis=1)]
print('hours with at least one null:', len(gap_rows))
gap_rows

hours with at least one null: 2


,ts_utc,Cross border electricity trading,Nuclear,Hydro water reservoir,Others,Wind onshore,Load,Residual load,Renewable share of load,Renewable share of generation,Fossil gas,Solar
10822,2022-03-27 21:00:00,-5791.7,NaN,NaN,NaN,NaN,14791.0,14791.0,NaN,NaN,NaN,NaN
34319,2024-11-30 22:00:00,-5664.0,5795.0,NaN,527.8,8184.7,14708.0,6522.4,55.7,56.4,0.4,0.9


In [59]:
before_fill = int(generation[value_names].isna().sum().sum())
generation[value_names] = generation[value_names].interpolate(method='linear', limit_direction='both')
after_fill = int(generation[value_names].isna().sum().sum())
print(f'interpolated {before_fill - after_fill} values across {len(gap_rows)} hours')
print(f'nulls remaining: {after_fill}')

interpolated 9 values across 2 hours
nulls remaining: 0


### Column names

The API labels are human-readable but not usable as identifiers, so they are mapped
to snake_case with the unit in the name. Any series that turns up without a mapping
is printed rather than silently dropped, which is what would happen if the rename
were followed by a blind column selection.

In [60]:
GEN_COLUMNS = {
    'Cross border electricity trading': 'cross_border_trading_mw',
    'Nuclear': 'nuclear_mw',
    'Fossil gas': 'fossil_gas_mw',
    'Hydro water reservoir': 'hydro_reservoir_mw',
    'Others': 'other_mw',
    'Wind onshore': 'wind_onshore_mw',
    'Solar': 'solar_mw',
    'Load': 'load_mw',
    'Residual load': 'residual_load_mw',
    'Renewable share of load': 'renewable_share_load_pct',
    'Renewable share of generation': 'renewable_share_generation_pct',
}
unmapped = [c for c in value_names if c not in GEN_COLUMNS]
print('series with no mapping:', unmapped)
generation = generation.rename(columns=GEN_COLUMNS)[['ts_utc'] + list(GEN_COLUMNS.values())]
print(generation.shape)
generation.head()

series with no mapping: []
(43824, 12)


,ts_utc,cross_border_trading_mw,nuclear_mw,fossil_gas_mw,hydro_reservoir_mw,other_mw,wind_onshore_mw,solar_mw,load_mw,residual_load_mw,renewable_share_load_pct,renewable_share_generation_pct
0,2020-12-31 23:00:00,-3979.5,6771.0,0.0,10552.0,1044.0,1030.0,0.0,15678.0,14648.0,73.9,59.7
1,2021-01-01 00:00:00,-4143.7,6773.0,0.0,10419.0,1038.0,978.0,0.0,15507.0,14529.0,73.5,59.3
2,2021-01-01 01:00:00,-4355.9,6772.0,0.0,10280.0,1010.0,913.0,0.0,15130.0,14217.0,74.0,59.0
3,2021-01-01 02:00:00,-4389.4,6775.0,0.0,10221.0,997.0,914.0,0.0,14917.0,14003.0,74.6,58.9
4,2021-01-01 03:00:00,-4519.3,6775.0,0.0,10302.0,998.0,959.0,0.0,15029.0,14070.0,74.9,59.2


### Sanity checks

Three numbers that would catch the wrong country or the wrong unit. Swedish nuclear
runs in the low thousands of MW, load sits roughly between 9,000 and 27,000 MW, and
cross-border trading should be negative in most hours because Sweden exports more
than it imports. German load, which is what a silently ignored `bzn` parameter would
have returned, is an order of magnitude larger.

In [61]:
print(generation[['nuclear_mw', 'load_mw', 'cross_border_trading_mw', 'wind_onshore_mw']].describe().round(0).to_string())
print()
print(f"nuclear median:  {generation['nuclear_mw'].median():>9,.0f} MW")
print(f"load range:      {generation['load_mw'].min():>9,.0f} to {generation['load_mw'].max():,.0f} MW")
print(f"net export hours: {(generation['cross_border_trading_mw'] < 0).mean():.1%}")

       nuclear_mw  load_mw  cross_border_trading_mw  wind_onshore_mw
count     43824.0  43824.0                  43824.0          43824.0
mean       5513.0  15141.0                  -3601.0           3941.0
std        1018.0   3282.0                   1534.0           2457.0
min        2584.0   8259.0                  -9170.0             83.0
25%        4590.0  12536.0                  -4636.0           1980.0
50%        5605.0  14628.0                  -3636.0           3465.0
75%        6360.0  17505.0                  -2600.0           5504.0
max       11394.0  25756.0                   2512.0          13528.0

nuclear median:      5,605 MW
load range:          8,259 to 25,756 MW
net export hours: 98.5%


In [62]:
gen_gaps = HOURLY_GRID.difference(pd.DatetimeIndex(generation['ts_utc']))
outside_gen = generation[(generation['ts_utc'] < HOURLY_GRID[0]) | (generation['ts_utc'] > HOURLY_GRID[-1])]
print(f'rows: {len(generation):,}')
print(f'missing hours against the {len(HOURLY_GRID):,} hour grid: {len(gen_gaps)}  {list(gen_gaps)}')
print(f'rows outside the grid, kept: {len(outside_gen)}')
generation['ts_utc'].dt.year.value_counts().sort_index()

rows: 43,824
missing hours against the 43,824 hour grid: 1  [Timestamp('2025-12-31 23:00:00')]
rows outside the grid, kept: 1


ts_utc
2020       1
2021    8760
2022    8760
2023    8760
2024    8784
2025    8759
Name: count, dtype: int64

In [63]:
generation.to_csv(CLEAN_DIR / 'clean_generation_hourly.csv', index=False)
print('wrote data/clean/clean_generation_hourly.csv', generation.shape)
generation.head()

wrote data/clean/clean_generation_hourly.csv (43824, 12)


,ts_utc,cross_border_trading_mw,nuclear_mw,fossil_gas_mw,hydro_reservoir_mw,other_mw,wind_onshore_mw,solar_mw,load_mw,residual_load_mw,renewable_share_load_pct,renewable_share_generation_pct
0,2020-12-31 23:00:00,-3979.5,6771.0,0.0,10552.0,1044.0,1030.0,0.0,15678.0,14648.0,73.9,59.7
1,2021-01-01 00:00:00,-4143.7,6773.0,0.0,10419.0,1038.0,978.0,0.0,15507.0,14529.0,73.5,59.3
2,2021-01-01 01:00:00,-4355.9,6772.0,0.0,10280.0,1010.0,913.0,0.0,15130.0,14217.0,74.0,59.0
3,2021-01-01 02:00:00,-4389.4,6775.0,0.0,10221.0,997.0,914.0,0.0,14917.0,14003.0,74.6,58.9
4,2021-01-01 03:00:00,-4519.3,6775.0,0.0,10302.0,998.0,959.0,0.0,15029.0,14070.0,74.9,59.2


---
# 6. Coverage and summary

How much of the reference grid each series actually covers. A missing hour here
means the grid has an hour for which the cleaned file has no row, whether because
the source never published it or because the series starts late.

In [64]:
price_gaps = missing_hours(prices_hourly)
temp_gaps = missing_hours(temperature)
coverage = pd.DataFrame({'price_hours_missing': price_gaps, 'temp_hours_missing': temp_gaps})
coverage['price_pct'] = (100 * coverage['price_hours_missing'] / len(HOURLY_GRID)).round(2)
coverage['temp_pct'] = (100 * coverage['temp_hours_missing'] / len(HOURLY_GRID)).round(2)
print(f'reference grid: {len(HOURLY_GRID):,} hours per zone')
coverage

reference grid: 43,824 hours per zone


,price_hours_missing,temp_hours_missing,price_pct,temp_pct
SE1,903,181,2.06,0.41
SE2,922,175,2.10,0.40
SE3,758,117,1.73,0.27
SE4,758,10,1.73,0.02


The price gaps are not one long outage. They are scattered across the whole period
at roughly five to twenty hours a month, with one clear spike in October 2025 that
lines up with the change in resolution. The temperature gaps are mostly the late
start of three of the four stations on 1 January 2021.

In [65]:
se1_gaps = HOURLY_GRID.difference(pd.DatetimeIndex(prices_hourly.loc[prices_hourly['zone'] == 'SE1', 'ts_utc']))
pd.Series(se1_gaps).dt.to_period('M').value_counts().sort_index().tail(15)

2024-10     13
2024-11     11
2024-12     17
2025-01     19
2025-02     13
2025-03     21
2025-04     12
2025-05     15
2025-06     14
2025-07     12
2025-08      9
2025-09     13
2025-10    195
2025-11      4
2025-12      2
Freq: M, Name: count, dtype: int64

In [66]:
temperature.groupby('zone')['ts_utc'].agg(['min', 'max', 'count'])

,min,max,count
zone,,,
SE1,2021-01-01 06:00:00,2025-12-31 23:00:00,43643
SE2,2021-01-01 06:00:00,2025-12-31 23:00:00,43649
SE3,2021-01-01 06:00:00,2025-12-31 23:00:00,43707
SE4,2021-01-01 00:00:00,2025-12-31 23:00:00,43814


In [67]:
summary_rows = []
summary_rows.append({'output': 'data/clean/clean_prices_hourly.csv', 'rows': len(prices_hourly), 'from': str(prices_hourly['ts_utc'].min()), 'to': str(prices_hourly['ts_utc'].max()), 'missing_vs_grid': '; '.join(f'{z}:{n}' for z, n in sorted(price_gaps.items()))})
summary_rows.append({'output': 'data/clean/clean_scb_monthly.csv', 'rows': len(scb_clean), 'from': str(scb_clean['month'].min().date()), 'to': str(scb_clean['month'].max().date()), 'missing_vs_grid': f'{len(scb_counts) * len(MONTHLY_GRID) - len(scb_clean)} missing month cells'})
summary_rows.append({'output': 'data/clean/clean_temperature_hourly.csv', 'rows': len(temperature), 'from': str(temperature['ts_utc'].min()), 'to': str(temperature['ts_utc'].max()), 'missing_vs_grid': '; '.join(f'{z}:{n}' for z, n in sorted(temp_gaps.items()))})
summary_rows.append({'output': 'data/clean/clean_flows_hourly.csv', 'rows': len(flows), 'from': str(flows['ts_utc'].min()), 'to': str(flows['ts_utc'].max()), 'missing_vs_grid': f'{len(flow_gaps)} missing (national series)'})
summary_rows.append({'output': 'data/clean/clean_generation_hourly.csv', 'rows': len(generation), 'from': str(generation['ts_utc'].min()), 'to': str(generation['ts_utc'].max()), 'missing_vs_grid': f'{len(gen_gaps)} missing (national series)'})
summary = pd.DataFrame(summary_rows)
summary

,output,rows,from,to,missing_vs_grid
0,data/clean/clean_prices_hourly.csv,171959,2020-12-31 23:00:00,2025-12-31 22:00:00,SE1:903; SE2:922; SE3:758; SE4:758
1,data/clean/clean_scb_monthly.csv,2880,2021-01-01,2025-12-01,0 missing month cells
2,data/clean/clean_temperature_hourly.csv,174813,2021-01-01 00:00:00,2025-12-31 23:00:00,SE1:181; SE2:175; SE3:117; SE4:10
3,data/clean/clean_flows_hourly.csv,43824,2020-12-31 23:00:00,2025-12-31 22:00:00,1 missing (national series)
4,data/clean/clean_generation_hourly.csv,43824,2020-12-31 23:00:00,2025-12-31 22:00:00,1 missing (national series)


### What was removed, and why

| Dataset | Rows in | Rows out | Removed |
|---|---|---|---|
| Spot prices | 194,804 | 171,959 hourly | 3 exact duplicate rows, then 194,801 rows averaged into hourly means |
| SCB monthly | 3,168 after unpivot | 2,880 | 288 rows for 2026M01 to 2026M06, outside the project period |
| SMHI temperature | 174,813 | 174,813 | nothing; no bad timestamps, no duplicates, no non-numeric readings |
| Cross-border flows | 70,104 across five files | 43,824 | nothing; the 2025 file is averaged from 15 minute steps to hourly |
| National generation and load | 43,824 across five files | 43,824 | nothing; 16,704 nulls zero-filled for 2021 solar and gas, 9 hours interpolated |

Two things deliberately **not** removed: the four price rows at 23:00 UTC on
31 December 2020, and the 33 temperature readings flagged `Y`. Both are reported
above and left in the output for a later step to decide on.

---
# 7. Usable panel size

Each cleaned file covers a slightly different set of hours, so the number that
matters before any modelling is the intersection: hours where everything needed is
present at the same time. The count is built up in four steps.

- **Price complete**: the hour has a non-missing price for all four zones.
- **Plus temperature**: the hour also has a non-missing temperature for all four
  stations.
- **Plus flows**: the hour also has a cross-border flow observation.
- **Plus generation**: the hour also has a national generation and load row.

Flows and generation are national, so one row of each covers all four zones at once.
The SCB file is monthly and joins on month rather than hour, so it does not enter
this count.

In [68]:
# Read the cleaned outputs back from disk, so this section checks the files that
# were actually written rather than the frames still in memory.
prices_out = pd.read_csv(CLEAN_DIR / 'clean_prices_hourly.csv', parse_dates=['ts_utc'])
temp_out = pd.read_csv(CLEAN_DIR / 'clean_temperature_hourly.csv', parse_dates=['ts_utc'])
flows_out = pd.read_csv(CLEAN_DIR / 'clean_flows_hourly.csv', parse_dates=['ts_utc'])
gen_out = pd.read_csv(CLEAN_DIR / 'clean_generation_hourly.csv', parse_dates=['ts_utc'])
print('prices     ', prices_out.shape)
print('temperature', temp_out.shape)
print('flows      ', flows_out.shape)
print('generation ', gen_out.shape)

prices      (171959, 3)
temperature (174813, 4)
flows       (43824, 8)
generation  (43824, 12)


In [69]:
ZONES = ['SE1', 'SE2', 'SE3', 'SE4']

price_per_hour = prices_out.dropna(subset=['price_eur_mwh']).groupby('ts_utc')['zone'].nunique()
price_hours = pd.DatetimeIndex(price_per_hour[price_per_hour == len(ZONES)].index)

temp_per_hour = temp_out.dropna(subset=['temp_c']).groupby('ts_utc')['zone'].nunique()
temp_hours = pd.DatetimeIndex(temp_per_hour[temp_per_hour == len(ZONES)].index)

flow_hours = pd.DatetimeIndex(flows_out.dropna(subset=['sum_gw'])['ts_utc'].unique())
gen_hours = pd.DatetimeIndex(gen_out.dropna(subset=['load_mw'])['ts_utc'].unique())

print(f'hours with a price for all four zones:       {len(price_hours):,}')
print(f'hours with a temperature for all four zones: {len(temp_hours):,}')
print(f'hours with a flow observation:               {len(flow_hours):,}')
print(f'hours with a generation and load row:        {len(gen_hours):,}')

hours with a price for all four zones:       41,678
hours with a temperature for all four zones: 43,470
hours with a flow observation:               43,824
hours with a generation and load row:        43,824


In [70]:
# Intersect step by step, all of it clipped to the reference grid.
step_price = HOURLY_GRID.intersection(price_hours)
step_temp = step_price.intersection(temp_hours)
step_flows = step_temp.intersection(flow_hours)
step_gen = step_flows.intersection(gen_hours)

panel = pd.DataFrame([
    {'step': 'reference grid', 'hours': len(HOURLY_GRID), 'lost_at_this_step': 0},
    {'step': 'price, all four zones', 'hours': len(step_price), 'lost_at_this_step': len(HOURLY_GRID) - len(step_price)},
    {'step': 'plus temperature, all four stations', 'hours': len(step_temp), 'lost_at_this_step': len(step_price) - len(step_temp)},
    {'step': 'plus cross-border flows', 'hours': len(step_flows), 'lost_at_this_step': len(step_temp) - len(step_flows)},
    {'step': 'plus generation and load', 'hours': len(step_gen), 'lost_at_this_step': len(step_flows) - len(step_gen)},
])
panel['share_of_grid_pct'] = (100 * panel['hours'] / len(HOURLY_GRID)).round(2)
panel

,step,hours,lost_at_this_step,share_of_grid_pct
0,reference grid,43824,0,100.00
1,"price, all four zones",41677,2147,95.10
2,"plus temperature, all four stations",41337,340,94.33
3,plus cross-border flows,41337,0,94.33
4,plus generation and load,41337,0,94.33


In [71]:
by_year = pd.DataFrame({
    'grid_hours': pd.Series(HOURLY_GRID.year).value_counts(),
    'price_only': pd.Series(step_price.year).value_counts(),
    'price_and_temp': pd.Series(step_temp.year).value_counts(),
    'plus_flows': pd.Series(step_flows.year).value_counts(),
    'all_four': pd.Series(step_gen.year).value_counts(),
}).fillna(0).astype(int).sort_index()
by_year.index.name = 'year'
by_year['all_four_pct'] = (100 * by_year['all_four'] / by_year['grid_hours']).round(1)
by_year

,grid_hours,price_only,price_and_temp,plus_flows,all_four,all_four_pct
year,,,,,,
2021,8760,8404,8260,8260,8260,94.3
2022,8760,8373,8322,8322,8322,95.0
2023,8760,8386,8311,8311,8311,94.9
2024,8784,8357,8325,8325,8325,94.8
2025,8760,8157,8119,8119,8119,92.7


The last column is the share of each year that survives all four joins. Every hour
dropped here is one where at least one series has no observation; nothing has been
filled, interpolated or carried forward at this stage. A model estimated on the
hourly panel has the `all_four` count to work with, by year or in total.